# Process Many Requests with the Batch API

This example categorizes products in the background. Use normal API requests when a user needs an immediate answer.

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import json

load_dotenv()
client = OpenAI()

In [2]:
products = [
    {
        "id": "product-101",
        "name": "Trail Running Shoes",
        "description": "Lightweight shoes for running on rocky paths."
    },
    {
        "id": "product-102",
        "name": "Wireless Headphones",
        "description": "Noise-cancelling headphones with Bluetooth."
    },
    {
        "id": "product-103",
        "name": "Ceramic Table Lamp",
        "description": "A small lamp for a bedside table or desk."
    }
]

## Immediate Result

A normal request returns one category immediately. This is useful when a user is waiting for the answer.

In [4]:
product = products[0]

response = client.responses.create(
    model="gpt-4o-mini",
    instructions="Choose one category for the given product out of Clothing, Electronics, Home, or Sports.",
    input=f"{product['name']}: {product['description']}"
)

print(response.output_text)

Sports


## Batch API

For a large product catalog, create a JSONL file with one request per line.

In [5]:
with open("product_batch.jsonl", "w") as batch_file:
    for product in products:
        request = {
            "custom_id": product["id"],
            "method": "POST",
            "url": "/v1/responses",
            "body": {
                "model": "gpt-4o-mini",
                "instructions": "Choose one category for the given product out of Clothing, Electronics, Home, or Sports.",
                "input": f"{product['name']}: {product['description']}"
            }
        }

        batch_file.write(json.dumps(request) + "\n")

In [6]:
with open("product_batch.jsonl", "rb") as batch_file:
    uploaded_file = client.files.create(
        file=batch_file,
        purpose="batch"
    )

print(uploaded_file.id)

file-NPrwBsKZCr1faA9xRFbzZd


In [16]:
batch = client.batches.create(
    input_file_id=uploaded_file.id,
    endpoint="/v1/responses",
    completion_window="24h"
)

print(batch.id)
print(batch.status)

batch_6ae148dc607c8190a9a94c42be2dfd51
validating


Run the next cell later. Download the output only after the status is `completed`.

In [27]:
batch = client.batches.retrieve(batch.id)
print(batch.status)

completed


In [28]:
if batch.status == "completed":
    file_response = client.files.content(batch.output_file_id)

    with open("product_batch_results.jsonl", "w") as output_file:
        output_file.write(file_response.text)

In [46]:
with open("product_batch_results.jsonl") as output_file:
    for line in output_file:
        result = json.loads(line)
        response = result["response"]
        body = response["body"]

        print(result["custom_id"], body["output"][0]["content"][0]["text"])
        

product-101 Sports
product-102 Electronics
product-103 Home
